In [4]:
from pathlib import Path

import onnxruntime as ort
import numpy as np

In [ ]:
# Settings
RUNS_DIR = Path.cwd().parent / "02-balance-bot-with-ppo" / "runs"
ONNX_MODELS = list(RUNS_DIR.glob("**/actor.onnx"))
if not ONNX_MODELS:
    raise FileNotFoundError(f"No actor.onnx found under {RUNS_DIR}. Run PPO training and export an actor first.")
ONNX_PATH = max(ONNX_MODELS, key=lambda path: path.stat().st_mtime)

# Test observation [pitch, pitch_rate, wheel_vel_left, wheel_vel_right]
TEST_OBS = np.array([[0.2, 1.0, -0.5, 0.5]], dtype=np.float32)

In [ ]:
# Load the ONNX model and inspect its input/output names
session = ort.InferenceSession(str(ONNX_PATH), providers=["CPUExecutionProvider"])
model_input = session.get_inputs()[0]
model_output = session.get_outputs()[0]

# Run one inference
action = session.run([model_output.name], {model_input.name: TEST_OBS})[0].reshape(-1)

# Print a compact, labeled summary
observation_labels = ["pitch", "pitch_rate", "left_wheel_velocity", "right_wheel_velocity"]
action_labels = ["left_motor", "right_motor"]
print("ONNX inference test")
print(f"Model: {ONNX_PATH}")
print(f"Input: {model_input.name} {tuple(model_input.shape)}")
print("Observation:")
for label, value in zip(observation_labels, TEST_OBS[0]):
    print(f"  {label:22s} {value:+.4f}")
print(f"Output: {model_output.name}")
print("Predicted action:")
for index, value in enumerate(action):
    label = action_labels[index] if index < len(action_labels) else f"action_{index}"
    print(f"  {label:22s} {value:+.4f}")
print("Inference only: this does not open MuJoCo or command hardware.")

[array([[0.30135354, 0.17316583]], dtype=float32)]
